In [1]:
import os
os.chdir("..")

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from collections import deque

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

# Loading data

In [22]:
df = pd.read_csv("data/merged/merged_v6.csv")
df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_convert("Europe/Kyiv")

In [23]:
df.head()

,region_id,time,alarm,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_preciptype,hour_windspeed,hour_winddir,hour_pressure,hour_visibility,hour_cloudcover,hour_uvindex,hour_conditions,region_city,messages_count,has_threat_sum,nlp_артобстрілу,nlp_бпла,nlp_відбій,nlp_відбій_тривоги,nlp_дніпропетровська,nlp_донецька,nlp_запорізька,nlp_нікополь,nlp_нікополь_нікопольська,nlp_нікопольська,nlp_повітряна,nlp_повітряна_тривога,nlp_тривога,nlp_тривоги,nlp_харківська,msg_count_last_3h,msg_count_last_24h,threat_diff_1h,day_of_week,is_weekend,text_length,isw_cluster_0,isw_cluster_1,isw_cluster_2,isw_cluster_3,isw_cluster_4,isw_cluster_5,isw_cluster_6,isw_cluster_7,isw_cluster_8,isw_cluster_9,anomaly_count_7d,avg_dist_centroid_7d,news_count_7d,topic_entropy_7d,topic_entropy_30d,news_velocity_30d,news_velocity_7d,centroid_shift_7d,avg_dist_centroid_30d,dom_cluster_share_7d,anomaly_count_30d,centroid_shift_30d,news_count_30d,dom_cluster_share_30d,year,month,day,hour
0,1,2022-03-28 00:00:00+03:00,0,3.0,0.4,32.87,-11.8,0.0,0.0,0,9.4,285.0,1030.6,20.0,0.0,0.0,0,21,13,5,0,0,4,4,0,0,0,0,0,0,5,5,5,4,0,86,289,-3,0,0,13953,0,0,0,0,0,0,0,0,0,1,1,0.345617,7,0.59827,0.589003,28,0,0.22995,0.396254,0.714286,3,0.565055,29,0.724138,2022,3,28,0
1,1,2022-03-28 01:00:00+03:00,0,2.9,-0.1,33.36,-11.7,0.0,0.0,0,10.8,305.0,1030.7,20.0,0.0,0.0,0,21,8,2,0,0,3,3,0,0,0,0,0,0,1,1,1,4,0,45,285,-3,0,0,13953,0,0,0,0,0,0,0,0,0,1,1,0.345617,7,0.59827,0.589003,28,0,0.22995,0.396254,0.714286,3,0.565055,29,0.724138,2022,3,28,1
2,1,2022-03-28 02:00:00+03:00,0,1.2,-1.0,41.88,-10.3,0.0,0.0,0,7.2,272.0,1031.3,20.0,11.1,0.0,0,21,7,0,0,0,1,1,0,0,0,0,0,0,0,0,0,1,0,28,269,-2,0,0,13953,0,0,0,0,0,0,0,0,0,1,1,0.345617,7,0.59827,0.589003,28,0,0.22995,0.396254,0.714286,3,0.565055,29,0.724138,2022,3,28,2
3,1,2022-03-28 03:00:00+03:00,0,0.4,-1.6,45.73,-10.0,0.0,0.0,0,6.0,358.0,1031.4,20.0,80.0,0.0,4,21,11,10,0,0,0,0,2,0,2,0,0,0,9,9,9,0,2,26,275,10,0,0,13953,0,0,0,0,0,0,0,0,0,1,1,0.345617,7,0.59827,0.589003,28,0,0.22995,0.396254,0.714286,3,0.565055,29,0.724138,2022,3,28,3
4,1,2022-03-28 04:00:00+03:00,0,0.0,-2.9,47.65,-9.8,0.0,0.0,0,8.7,282.0,1031.1,20.0,90.0,0.0,4,21,9,1,0,0,6,6,2,0,2,0,0,0,1,1,2,6,2,27,283,-9,0,0,13953,0,0,0,0,0,0,0,0,0,1,1,0.345617,7,0.59827,0.589003,28,0,0.22995,0.396254,0.714286,3,0.565055,29,0.724138,2022,3,28,4


In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 805089 entries, 0 to 805088
Data columns (total 69 columns):
 #   Column                     Non-Null Count   Dtype                      
---  ------                     --------------   -----                      
 0   region_id                  805089 non-null  int64                      
 1   time                       805089 non-null  datetime64[us, Europe/Kyiv]
 2   alarm                      805089 non-null  int64                      
 3   hour_temp                  805089 non-null  float64                    
 4   hour_feelslike             805089 non-null  float64                    
 5   hour_humidity              805089 non-null  float64                    
 6   hour_dew                   805089 non-null  float64                    
 7   hour_precip                805089 non-null  float64                    
 8   hour_precipprob            805089 non-null  float64                    
 9   hour_preciptype            805089 non-null  int6

# Extracting features

Calculate distance in regions to nearest region with alarm

In [25]:
%%time
# Neighbor adjacency map using region_ids from your CSV:
# 1=Crimea, 2=Vinnytsia, 3=Volyn, 4=Dnipropetrovsk, 5=Donetsk,
# 6=Zhytomyr, 7=Zakarpattia, 8=Zaporizhzhia, 9=Ivano-Frankivsk,
# 10=Kyiv, 11=Kirovohrad, 12=Luhansk, 13=Lviv, 14=Mykolaiv,
# 15=Odessa, 16=Poltava, 17=Rivne, 18=Sumy, 19=Ternopil,
# 20=Kharkiv, 21=Kherson, 22=Khmelnytskyi, 23=Cherkasy,
# 24=Chernivtsi, 25=Chernihiv

NEIGHBORS = {
    1:  [8, 14, 15],
    2:  [6, 10, 23, 11, 15, 24, 22],
    3:  [17, 13],
    4:  [16, 20, 8, 21, 14, 11, 5],
    5:  [20, 12, 8, 4],
    6:  [10, 25, 17, 22, 2],
    7:  [13, 9],
    8:  [4, 5, 21],
    9:  [13, 7, 24, 19],
    10: [25, 18, 16, 23, 2, 6],
    11: [23, 16, 4, 14, 15, 2],
    12: [20, 5],
    13: [3, 17, 19, 9, 7],
    14: [15, 21, 4, 11],
    15: [14, 11, 2, 24],
    16: [18, 20, 4, 11, 23, 25, 10],
    17: [3, 13, 19, 22, 6],
    18: [25, 16, 20],
    19: [17, 22, 13, 9, 24],
    20: [18, 16, 4, 5, 12],
    21: [14, 4, 8],
    22: [6, 2, 19, 17, 24],
    23: [10, 16, 4, 11, 2, 25],
    24: [9, 22, 2, 15, 19],
    25: [10, 18, 16, 23],
}

# Precompute BFS distances between all region pairs once at import time
def _bfs_distances(source, neighbors_map):
    """Returns dict of {region_id: hops} from source via BFS."""
    dist = {source: 0}
    queue = deque([source])
    while queue:
        node = queue.popleft()
        for neighbor in neighbors_map[node]:
            if neighbor not in dist:
                dist[neighbor] = dist[node] + 1
                queue.append(neighbor)
    return dist

ALL_GRAPH_DISTANCES = {
    r: _bfs_distances(r, NEIGHBORS)
    for r in NEIGHBORS
}

def dist_to_nearest_alarm(row, alarm_lookup, region_col="region_id", time_col="time", alarm_col="alarm"):
    """
    For use with df.apply(). Returns:
      0    — current region has an alarm
      hops — min number of borders to cross to reach a region with alarm
      -1   — no alarms anywhere at this timestamp
    
    Parameters:
        row          - DataFrame row (from apply axis=1)
        alarm_lookup - dict mapping (timestamp, region_id) -> alarm value;
                       build it once before calling apply (see usage below)
        region_col   - name of the region ID column
        time_col     - name of the timestamp column
        alarm_col    - name of the alarm status column
    """
    current_region = row[region_col]
    current_time   = row[time_col]

    # Current region has alarm → distance is 0
    if row[alarm_col]:
        return 0

    distances_from_current = ALL_GRAPH_DISTANCES[current_region]

    min_hops = float("inf")
    for region_id in NEIGHBORS:
        if region_id == current_region:
            continue
        try:
            has_alarm = alarm_lookup[(current_time, region_id)]
        except KeyError:
            continue
        if has_alarm:
            hops = distances_from_current.get(region_id, float("inf"))
            if hops < min_hops:
                min_hops = hops

    return -1 if min_hops == float("inf") else min_hops


# --- Usage ---

# Build lookup ONCE before apply (much faster than indexing df inside apply)
alarm_lookup = df.set_index(["time", "region_id"])["alarm"].to_dict()

df["hops_to_nearest_alarm"] = df.apply(
    dist_to_nearest_alarm,
    axis=1,
    alarm_lookup=alarm_lookup,
    region_col="region_id",
    time_col="time",
    alarm_col="alarm",
)

df["hops_to_nearest_alarm"] = df.groupby("region_id")["hops_to_nearest_alarm"].shift(1)

CPU times: total: 25.7 s
Wall time: 26.7 s


In [26]:
df.hops_to_nearest_alarm.unique()

array([nan,  1.,  3.,  4., -1.,  0.,  2.,  5.,  6.,  7.])

Calculate number of alarms in all regions by hour

In [27]:
all_alarms = df.groupby("time")["alarm"].sum().rename("num_alarms").reset_index()

df = pd.merge(df, all_alarms, on="time", how="left")
# df["other_alarms_count"] = df["num_alarms"] - df["alarm"]
for i in range(12):
    df[f"alarms_count_{i+1}h_ago"] = df["num_alarms"].shift(i+1)

df = df.drop(columns="num_alarms")

Add lag for 1-24 hours

In [28]:
for i in range(24):
    df[f"alarm_status_{i+1}h_ago"] = df.groupby("region_id")["alarm"].shift(i+1)

In [29]:
df = df.dropna(axis=0).reset_index(drop=True)

In [30]:
df.head()

,region_id,time,alarm,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_precipprob,hour_preciptype,hour_windspeed,hour_winddir,hour_pressure,hour_visibility,hour_cloudcover,hour_uvindex,hour_conditions,region_city,messages_count,has_threat_sum,nlp_артобстрілу,nlp_бпла,nlp_відбій,nlp_відбій_тривоги,nlp_дніпропетровська,nlp_донецька,nlp_запорізька,nlp_нікополь,nlp_нікополь_нікопольська,nlp_нікопольська,nlp_повітряна,nlp_повітряна_тривога,nlp_тривога,nlp_тривоги,nlp_харківська,msg_count_last_3h,msg_count_last_24h,threat_diff_1h,day_of_week,is_weekend,text_length,isw_cluster_0,isw_cluster_1,isw_cluster_2,isw_cluster_3,isw_cluster_4,isw_cluster_5,isw_cluster_6,isw_cluster_7,isw_cluster_8,isw_cluster_9,anomaly_count_7d,avg_dist_centroid_7d,news_count_7d,topic_entropy_7d,topic_entropy_30d,news_velocity_30d,news_velocity_7d,centroid_shift_7d,avg_dist_centroid_30d,dom_cluster_share_7d,anomaly_count_30d,centroid_shift_30d,news_count_30d,dom_cluster_share_30d,year,month,day,hour,hops_to_nearest_alarm,alarms_count_1h_ago,alarms_count_2h_ago,alarms_count_3h_ago,alarms_count_4h_ago,alarms_count_5h_ago,alarms_count_6h_ago,alarms_count_7h_ago,alarms_count_8h_ago,alarms_count_9h_ago,alarms_count_10h_ago,alarms_count_11h_ago,alarms_count_12h_ago,alarm_status_1h_ago,alarm_status_2h_ago,alarm_status_3h_ago,alarm_status_4h_ago,alarm_status_5h_ago,alarm_status_6h_ago,alarm_status_7h_ago,alarm_status_8h_ago,alarm_status_9h_ago,alarm_status_10h_ago,alarm_status_11h_ago,alarm_status_12h_ago,alarm_status_13h_ago,alarm_status_14h_ago,alarm_status_15h_ago,alarm_status_16h_ago,alarm_status_17h_ago,alarm_status_18h_ago,alarm_status_19h_ago,alarm_status_20h_ago,alarm_status_21h_ago,alarm_status_22h_ago,alarm_status_23h_ago,alarm_status_24h_ago
0,1,2022-03-29 00:00:00+03:00,0,11.6,11.6,27.80,-6.4,0.0,0.0,0,13.9,257.0,1016.0,20.0,0.0,0.0,0,21,13,0,0,0,8,8,0,0,0,0,0,0,0,0,2,8,0,51,337,-12,1,0,10309,0,0,0,1,0,0,0,0,0,0,1,0.340311,7,0.410116,0.589003,27,0,0.201195,0.393526,0.857143,3,0.527332,29,0.724138,2022,3,29,0,1.0,10.0,8.0,7.0,6.0,0.0,1.0,4.0,4.0,3.0,1.0,16.0,16.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,2022-03-29 01:00:00+03:00,0,11.0,11.0,29.92,-6.0,0.0,0.0,0,9.5,256.0,1015.8,20.0,20.7,0.0,4,21,6,1,0,0,4,4,0,0,2,0,0,0,1,1,2,4,0,41,335,1,1,0,10309,0,0,0,1,0,0,0,0,0,0,1,0.340311,7,0.410116,0.589003,27,0,0.201195,0.393526,0.857143,3,0.527332,29,0.724138,2022,3,29,1,1.0,9.0,10.0,8.0,7.0,6.0,0.0,1.0,4.0,4.0,3.0,1.0,16.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1,2022-03-29 02:00:00+03:00,0,10.3,10.3,32.92,-5.3,0.0,0.0,0,12.3,240.0,1015.3,20.0,30.6,0.0,4,21,1,1,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,20,329,0,1,0,10309,0,0,0,1,0,0,0,0,0,0,1,0.340311,7,0.410116,0.589003,27,0,0.201195,0.393526,0.857143,3,0.527332,29,0.724138,2022,3,29,2,1.0,4.0,9.0,10.0,8.0,7.0,6.0,0.0,1.0,4.0,4.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,2022-03-29 03:00:00+03:00,1,9.6,7.8,34.71,-5.3,0.0,0.0,0,12.3,241.0,1014.9,20.0,0.0,0.0,0,21,32,26,0,0,4,4,2,0,4,0,0,0,26,26,26,4,2,39,350,25,1,0,10309,0,0,0,1,0,0,0,0,0,0,1,0.340311,7,0.410116,0.589003,27,0,0.201195,0.393526,0.857143,3,0.527332,29,0.724138,2022,3,29,3,3.0,1.0,4.0,9.0,10.0,8.0,7.0,6.0,0.0,1.0,4.0,4.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1,2022-03-29 04:00:00+03:00,1,8.5,7.1,36.42,-5.6,0.0,0.0,0,8.6,217.0,1014.3,20.0,0.0,0.0,0,21,24,2,0,0,21,21,2,0,0,0,0,0,2,2,4,21,4,57,365,-24,1,0,10309,0,0,0,1,0,0,0,0,0,0,1,0.340311,7,0.410116,0.589003,27,0,0.201195,0.393526,0.857143,3,0.527332,29,0.724138,2022,3,29,4,0.0,21.0,1.0,4.0,9.0,10.0,8.0,7.0,6.0,0.0,1.0,4.0,4.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [31]:
df.to_csv("data/merged/merged_preprocessed.csv", index=False)

# Explorary data analysis

In [ ]:
df = pd.read_csv("data/merged/merged_preprocessed.csv")
df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_convert("Europe/Kyiv")